# Pipeline Completo — Monitor de Gastos Públicos Municipais
**LuxVerso Research Initiative**

## O que este notebook faz

Coleta, classifica e integra três fontes de dados públicos para gerar um dataset fiscal municipal enriquecido:

| Etapa | Fonte | O que traz |
|---|---|---|
| 1 | PNCP | Contratos públicos municipais (já coletados) |
| 2 | IBGE | População, PIB, área, densidade |
| 3 | SICONFI / Tesouro Nacional | Receita Corrente Líquida (RCL) |
| 4 | Merge | Dataset final com indicadores fiscais |

## Indicadores gerados

- `gasto_per_capita` — gasto artístico por habitante
- `gasto_pct_pib` — gasto como % do PIB municipal
- `gasto_pct_rcl` — **gasto como % da Receita Corrente Líquida** (mais poderoso)

## Pré-requisitos

Suba no Colab os dois arquivos antes de rodar:
- `pncp_bahia_2025_classificado.csv`
- `pncp_bahia_2026_classificado.csv`

## Erros conhecidos e corrigidos nesta versão

- ✓ SICONFI: endpoint correto é `/rreo` (não `/finbra` nem `/rgf`)
- ✓ SICONFI: filtro correto é `cod_conta = RREO3ReceitaCorrenteLiquida` + `coluna = TOTAL (ÚLTIMOS 12 MESES)`
- ✓ IBGE área: via GeoJSON (API SIDRA instável para esse dado)
- ✓ Saída: `.ipynb` e `.csv` (não `.zip`)
- ✓ Merge: chave `unidadeOrgao.codigoIbge` → `codigo_ibge`
- ✓ Rate limit SICONFI: 1 requisição/segundo

---
**Para rodar outro estado:** mude `UF_CODIGO` e `UF_SIGLA` na célula CONFIG.

---
## ETAPA 0 — Configuração

In [ ]:
import csv, json, os, time, urllib.request, urllib.error
from collections import defaultdict

# ── Mude aqui para outro estado ──────────────────
UF_CODIGO  = "29"    # 29 = Bahia
UF_SIGLA   = "BA"
ANO_SICONFI = 2023   # último ano completo disponível
OUTPUT_DIR = "output"
# ─────────────────────────────────────────────────

os.makedirs(OUTPUT_DIR, exist_ok=True)

IBGE_BASE    = "https://servicodados.ibge.gov.br/api"
SICONFI_BASE = "https://apidatalake.tesouro.gov.br/ords/siconfi/tt"

print(f"✓ Config: UF {UF_CODIGO} ({UF_SIGLA}) | SICONFI ano {ANO_SICONFI}")

---
## ETAPA 0B — Helper de requisição (retry + gzip)

In [ ]:
import gzip

def get_json(url, timeout=60, tentativas=3):
    """GET robusto com retry automático e suporte a gzip."""
    for i in range(tentativas):
        try:
            req = urllib.request.Request(
                url,
                headers={
                    "User-Agent": "pipeline-municipios/3.0",
                    "Accept-Encoding": "gzip"
                }
            )
            with urllib.request.urlopen(req, timeout=timeout) as r:
                raw = r.read()
                if r.info().get("Content-Encoding") == "gzip":
                    raw = gzip.decompress(raw)
                return json.loads(raw.decode("utf-8"))
        except urllib.error.HTTPError as e:
            print(f"  [HTTP {e.code}] tentativa {i+1}")
            if e.code == 404: return None
            time.sleep(2 ** i)
        except Exception as e:
            print(f"  [ERRO tentativa {i+1}] {type(e).__name__}: {e}")
            time.sleep(2 ** i)
    return None

def parse_sidra(data):
    """Extrai {codigo_ibge: valor_str} de resposta SIDRA padrão."""
    result = {}
    if not data: return result
    try:
        for item in data[0]["resultados"][0]["series"]:
            cod = item["localidade"]["id"]
            val = next((v for v in item["serie"].values() if v and v != "-"), None)
            if val:
                result[cod] = val
    except (KeyError, IndexError):
        pass
    return result

print("✓ Helpers prontos")

---
## ETAPA 1 — Carrega dados PNCP (já coletados e classificados)

In [ ]:
# Os arquivos PNCP já foram coletados e classificados anteriormente.
# Suba os dois CSVs no Colab antes de rodar esta célula.

pncp_files = [
    f"pncp_bahia_2025_classificado.csv",
    f"pncp_bahia_2026_classificado.csv",
]

pncp_rows = []
for fname in pncp_files:
    try:
        with open(fname, encoding="utf-8") as f:
            batch = list(csv.DictReader(f))
            pncp_rows.extend(batch)
        print(f"✓ {fname}: {len(batch)} linhas")
    except FileNotFoundError:
        print(f"✗ {fname} não encontrado — suba o arquivo no Colab")

print(f"\nTotal PNCP: {len(pncp_rows)} contratos")

# Distribuição por categoria
cats = defaultdict(int)
for r in pncp_rows:
    cats[r.get("categoria", "")] += 1
print("\nCategorias:")
for k, v in sorted(cats.items(), key=lambda x: -x[1]):
    if k:
        print(f"  {k}: {v}")

---
## ETAPA 2 — Coleta IBGE: municípios, população, PIB, área

In [ ]:
# ── 2A: Lista oficial de municípios ──────────────────────────────────
url = f"{IBGE_BASE}/v1/localidades/estados/{UF_CODIGO}/municipios"
data = get_json(url)
municipios = {str(m["id"]): m["nome"] for m in (data or [])}
print(f"✓ {len(municipios)} municípios em {UF_SIGLA}")

# ── 2B: População estimada 2024 ───────────────────────────────────────
# Tabela 6579, variável 9324
url = (f"{IBGE_BASE}/v3/agregados/6579/periodos/2024/variaveis/9324"
       f"?localidades=N6[N3[{UF_CODIGO}]]")
pop_raw = parse_sidra(get_json(url))
populacao = {}
for cod, val in pop_raw.items():
    try: populacao[cod] = int(val.replace(".", "").replace(",", ""))
    except: pass
print(f"✓ {len(populacao)} municípios com população")

# ── 2C: PIB municipal 2021 ────────────────────────────────────────────
# Tabela 5938, variável 37 (R$ mil → multiplicar por 1000)
url = (f"{IBGE_BASE}/v3/agregados/5938/periodos/2021/variaveis/37"
       f"?localidades=N6[N3[{UF_CODIGO}]]")
pib_raw = parse_sidra(get_json(url))
pib = {}
for cod, val in pib_raw.items():
    try: pib[cod] = float(val.replace(".", "").replace(",", ".")) * 1_000
    except: pass
print(f"✓ {len(pib)} municípios com PIB (2021)")

# ── 2D: Área territorial via GeoJSON ─────────────────────────────────
# Nota: API SIDRA é instável para área — usamos GeoJSON como fonte confiável
import math

url = f"https://raw.githubusercontent.com/tbrugz/geodata-br/master/geojson/geojs-{UF_CODIGO}-mun.json"
geo_data = get_json(url)

def polygon_area_km2(coords):
    area = 0
    for i in range(len(coords)):
        x1, y1 = coords[i]
        x2, y2 = coords[(i+1) % len(coords)]
        area += x1 * y2 - x2 * y1
    return abs(area) / 2 * 111 * 111  # graus² → km² (aproximação planar)

area = {}
if geo_data:
    for f in geo_data["features"]:
        cod = f["properties"]["id"]
        geom = f["geometry"]
        total = 0
        if geom["type"] == "Polygon":
            for ring in geom["coordinates"]:
                total += polygon_area_km2(ring)
        elif geom["type"] == "MultiPolygon":
            for poly in geom["coordinates"]:
                for ring in poly:
                    total += polygon_area_km2(ring)
        area[cod] = round(total, 2)
print(f"✓ {len(area)} municípios com área territorial (GeoJSON)")

---
## ETAPA 2B — Consolida e salva dataset IBGE

In [ ]:
ibge_dataset = []
ibge_cache = {}

for cod, nome in municipios.items():
    pop   = populacao.get(cod)
    pib_v = pib.get(cod)
    area_v = area.get(cod)
    pib_pc    = round(pib_v / pop)        if pib_v and pop   else None
    densidade = round(pop / area_v, 2)    if pop   and area_v else None

    row = {
        "codigo_ibge":       cod,
        "municipio":         nome,
        "uf":                UF_SIGLA,
        "populacao_2024":    pop,
        "pib_2021_R$":       round(pib_v) if pib_v else None,
        "pib_per_capita":    pib_pc,
        "area_km2":          area_v,
        "densidade_hab_km2": densidade,
    }
    ibge_dataset.append(row)
    ibge_cache[cod] = row

# Cobertura
total = len(ibge_dataset)
print(f"Municípios: {total}")
print(f"Com população: {sum(1 for r in ibge_dataset if r['populacao_2024'])} ({100*sum(1 for r in ibge_dataset if r['populacao_2024'])//total}%)")
print(f"Com PIB:       {sum(1 for r in ibge_dataset if r['pib_2021_R$'])} ({100*sum(1 for r in ibge_dataset if r['pib_2021_R$'])//total}%)")
print(f"Com área:      {sum(1 for r in ibge_dataset if r['area_km2'])} ({100*sum(1 for r in ibge_dataset if r['area_km2'])//total}%)")

# Salva CSV
ibge_path = f"{OUTPUT_DIR}/ibge_municipios_{UF_SIGLA}.csv"
with open(ibge_path, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(ibge_dataset[0].keys()))
    w.writeheader()
    w.writerows(ibge_dataset)
print(f"\n✓ {ibge_path}")

# Salva cache JSON (para lookup rápido)
cache_path = f"{OUTPUT_DIR}/ibge_cache_{UF_SIGLA}.json"
with open(cache_path, "w", encoding="utf-8") as f:
    json.dump(ibge_cache, f, ensure_ascii=False, indent=2)
print(f"✓ {cache_path}")

---
## ETAPA 3 — Coleta SICONFI: Receita Corrente Líquida (RCL)

**Fonte:** API DataLake do Tesouro Nacional — pública, sem chave, sem cadastro.

**Endpoint correto:** `/ords/siconfi/tt/rreo`

**Parâmetros:**
- `an_exercicio=2023`
- `nr_periodo=6` (6º bimestre = fechamento do ano)
- `co_tipo_demonstrativo=RREO`
- `no_anexo=RREO-Anexo 03` (Demonstrativo da RCL — Lei de Responsabilidade Fiscal)

**Filtro correto na resposta:**
- `cod_conta == "RREO3ReceitaCorrenteLiquida"`
- `coluna == "TOTAL (ÚLTIMOS 12 MESES)"`

**Rate limit:** 1 requisição/segundo (respeitado com `time.sleep(1.05)`)

In [ ]:
def fetch_rcl(codigo_ibge, ano=ANO_SICONFI):
    """
    Busca RCL via RREO Anexo 03.
    Retorna (valor_float, fonte_str) ou (None, motivo_str).
    """
    # Tenta RREO padrão (municípios >= 50k hab ou que optaram pelo padrão)
    for tipo in ["RREO", "RREO Simplificado"]:
        url = (
            f"{SICONFI_BASE}/rreo"
            f"?an_exercicio={ano}"
            f"&nr_periodo=6"
            f"&co_tipo_demonstrativo={urllib.parse.quote(tipo)}"
            f"&no_anexo=RREO-Anexo%2003"
            f"&id_ente={codigo_ibge}"
        )
        data = get_json(url, timeout=45)
        if not data or not data.get("items"):
            continue
        for item in data["items"]:
            if (item.get("cod_conta") == "RREO3ReceitaCorrenteLiquida"
                    and item.get("coluna") == "TOTAL (ÚLTIMOS 12 MESES)"):
                try:
                    return float(item["valor"]), tipo
                except (TypeError, ValueError):
                    pass
    return None, "sem_dados"

import urllib.parse

# ── Coleta para todos os municípios ──────────────────────────────────
# Tempo estimado: ~8 min (417 requisições × 1s)
siconfi_data = {}
erros = []
codes = list(municipios.keys())
total = len(codes)

print(f"Coletando RCL para {total} municípios...")
print("(~8 minutos — não feche o Colab)\n")

for i, cod in enumerate(codes):
    rcl, fonte = fetch_rcl(cod)
    siconfi_data[cod] = {
        "codigo_ibge": cod,
        "municipio":   municipios[cod],
        "uf":          UF_SIGLA,
        "ano":         ANO_SICONFI,
        "rcl_R$":      round(rcl) if rcl else None,
        "fonte_rcl":   fonte,
    }
    if not rcl:
        erros.append((cod, municipios[cod]))

    if (i + 1) % 50 == 0 or (i + 1) == total:
        ok = sum(1 for v in siconfi_data.values() if v["rcl_R$"])
        print(f"  [{i+1}/{total}] ✓ {ok} com RCL | ✗ {len(erros)} sem dados")

    time.sleep(1.05)  # respeita rate limit da API

print(f"\nFinalizado: {sum(1 for v in siconfi_data.values() if v['rcl_R$'])}/{total} com RCL")
if erros:
    print(f"Sem dados ({len(erros)}):")
    for cod, nome in erros[:10]:
        print(f"  {cod} — {nome}")

---
## ETAPA 3B — Salva dataset SICONFI

In [ ]:
rows_sic = list(siconfi_data.values())

siconfi_path = f"{OUTPUT_DIR}/siconfi_rcl_{UF_SIGLA}.csv"
with open(siconfi_path, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(rows_sic[0].keys()))
    w.writeheader()
    w.writerows(rows_sic)
print(f"✓ {siconfi_path} → {len(rows_sic)} linhas")

# Preview top 10
com_rcl = sorted([r for r in rows_sic if r["rcl_R$"]], key=lambda x: -x["rcl_R$"])
print("\nTop 10 maiores RCL:")
for r in com_rcl[:10]:
    print(f"  {r['municipio']:<25} R$ {r['rcl_R$']/1e6:>8.1f}M")

---
## ETAPA 4 — Merge final: PNCP + IBGE + SICONFI

**Chave de cruzamento:** `unidadeOrgao.codigoIbge` (PNCP) → `codigo_ibge` (IBGE/SICONFI)

**Indicadores calculados por contrato:**
- `gasto_per_capita` = valor / população
- `gasto_pct_pib` = valor / PIB × 100
- `gasto_pct_rcl` = valor / RCL × 100

In [ ]:
final_rows = []
sem_ibge = 0
sem_siconfi = 0

for r in pncp_rows:
    # Chave de join — campo do PNCP
    cod = r.get("unidadeOrgao.codigoIbge", "")

    ibge  = ibge_cache.get(cod, {})
    sicon = siconfi_data.get(cod, {})

    if not ibge:  sem_ibge   += 1
    if not sicon: sem_siconfi += 1

    pop   = ibge.get("populacao_2024")
    pib_v = ibge.get("pib_2021_R$")
    pib_pc = ibge.get("pib_per_capita")
    area_v = ibge.get("area_km2")
    dens  = ibge.get("densidade_hab_km2")
    rcl   = sicon.get("rcl_R$")

    try: valor = float(r.get("valor_final", 0) or 0)
    except: valor = 0.0

    final_rows.append({
        **r,
        # IBGE
        "ibge_populacao":   pop,
        "ibge_pib":         pib_v,
        "ibge_pib_pc":      pib_pc,
        "ibge_area_km2":    area_v,
        "ibge_densidade":   dens,
        # SICONFI
        "rcl_2023_R$":      rcl,
        # Indicadores
        "gasto_per_capita": round(valor / pop,   4) if pop   and pop   > 0 else None,
        "gasto_pct_pib":    round(valor / pib_v * 100, 6) if pib_v and pib_v > 0 else None,
        "gasto_pct_rcl":    round(valor / rcl   * 100, 6) if rcl   and rcl   > 0 else None,
    })

out_path = f"{OUTPUT_DIR}/pncp_ibge_siconfi_{UF_SIGLA}.csv"
with open(out_path, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(final_rows[0].keys()), extrasaction="ignore")
    w.writeheader()
    w.writerows(final_rows)

print(f"✓ {out_path} → {len(final_rows)} linhas")
if sem_ibge:    print(f"⚠ {sem_ibge} contratos sem match IBGE")
if sem_siconfi: print(f"⚠ {sem_siconfi} contratos sem match SICONFI")

print("\nColunas novas adicionadas:")
for col in ["ibge_populacao","ibge_pib","ibge_pib_pc","ibge_area_km2",
            "ibge_densidade","rcl_2023_R$","gasto_per_capita","gasto_pct_pib","gasto_pct_rcl"]:
    print(f"  {col}")

---
## ETAPA 5 — Rankings analíticos

In [ ]:
# Filtra contratos artísticos limpos
artisticos = [
    r for r in final_rows
    if r.get("categoria") == "artistico"
    and str(r.get("flag_outlier","")).lower()       not in ("true","1")
    and str(r.get("flag_nao_municipal","")).lower() not in ("true","1")
    and float(r.get("valor_final", 0) or 0) > 0
]
print(f"Contratos artísticos limpos: {len(artisticos)}")

# Agrega por município
mun = defaultdict(lambda: {
    "valor":0.0,"contratos":0,
    "pop":None,"pib":None,"rcl":None
})
for r in artisticos:
    nome = r.get("unidadeOrgao.municipioNome","")
    mun[nome]["valor"]     += float(r.get("valor_final",0) or 0)
    mun[nome]["contratos"] += 1
    mun[nome]["pop"]  = r.get("ibge_populacao")
    mun[nome]["pib"]  = r.get("ibge_pib")
    mun[nome]["rcl"]  = r.get("rcl_2023_R$")

ranking = []
for nome, v in mun.items():
    def safe_float(x):
        try: return float(x) if x else None
        except: return None
    pop_f = safe_float(v["pop"])
    pib_f = safe_float(v["pib"])
    rcl_f = safe_float(v["rcl"])
    ranking.append({
        "municipio":    nome,
        "valor_total":  round(v["valor"]),
        "contratos":    v["contratos"],
        "populacao":    int(pop_f) if pop_f else None,
        "per_capita":   round(v["valor"]/pop_f,2) if pop_f else None,
        "pct_pib":      round(v["valor"]/pib_f*100,3) if pib_f else None,
        "pct_rcl":      round(v["valor"]/rcl_f*100,3) if rcl_f else None,
        "pib_R$M":      round(pib_f/1e6,1)          if pib_f else None,
        "rcl_R$M":      round(rcl_f/1e6,1)          if rcl_f else None,
    })

rank_val = sorted(ranking, key=lambda x: -x["valor_total"])
rank_pc  = sorted([r for r in ranking if r["per_capita"]], key=lambda x: -x["per_capita"])
rank_rcl = sorted([r for r in ranking if r["pct_rcl"]],   key=lambda x: -x["pct_rcl"])

# Salva ranking completo
rank_path = f"{OUTPUT_DIR}/ranking_artistico_completo_{UF_SIGLA}.csv"
with open(rank_path, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(rank_val[0].keys()))
    w.writeheader()
    w.writerows(rank_val)
print(f"✓ {rank_path}")

# ── IMPRIME ────────────────────────────────────────────────────────────
print()
print("="*70)
print("TOP 15 — GASTO TOTAL (2025 + jan-abr/26)")
print("="*70)
for r in rank_val[:15]:
    print(f"  {r['municipio']:<25} R$ {r['valor_total']/1e6:>7.2f}M  ({r['contratos']} contratos)")

print()
print("="*70)
print("TOP 15 — PER CAPITA")
print("="*70)
for r in rank_pc[:15]:
    print(f"  {r['municipio']:<25} R$ {r['per_capita']:>8.2f}/hab  R$ {r['valor_total']/1e6:>5.2f}M")

print()
print("="*70)
print("TOP 15 — % DA RECEITA CORRENTE LÍQUIDA (mais explosivo)")
print("="*70)
for r in rank_rcl[:15]:
    print(f"  {r['municipio']:<25} {r['pct_rcl']:>7.3f}%  R$ {r['valor_total']/1e6:>5.2f}M  RCL=R$ {r['rcl_R$M']:>6.1f}M")

print()
print("✓ Pipeline completo!")
print(f"\nArquivos gerados em /{OUTPUT_DIR}/:")
for fname in [
    f"ibge_municipios_{UF_SIGLA}.csv",
    f"ibge_cache_{UF_SIGLA}.json",
    f"siconfi_rcl_{UF_SIGLA}.csv",
    f"pncp_ibge_siconfi_{UF_SIGLA}.csv",
    f"ranking_artistico_completo_{UF_SIGLA}.csv",
]:
    path = f"{OUTPUT_DIR}/{fname}"
    exists = "✓" if os.path.exists(path) else "✗"
    print(f"  {exists} {fname}")